# FPGA Camera Setup

This notebook documents the baseline OV7670 camera implementation used with the ZedBoard **before dataset-specific preprocessing or neural-network hardware was added**.

The purpose of this stage was to establish a reusable camera path that could:

1. configure the OV7670;
2. capture the parallel camera output;
3. store captured pixels in FPGA Block RAM; and
4. display the stored frame through the ZedBoard VGA output.

Later dataset implementations build on this baseline. MNIST-specific ROI selection, downsampling, thresholding, processed-image storage, and zyNet integration are intentionally outside the scope of this notebook.

## 1. Hardware and development environment

The camera-only project used:

- **ZedBoard Rev E**
- **Xilinx Zynq-7000 XC7Z020CLG484-1**
- **OV7670 camera module**
- **VGA display**
- **Vivado 2018.1**
- jumper-wire connections between the camera module and ZedBoard headers

This baseline uses programmable-logic resources and Block RAM. It does not depend on Zynq PS DDR memory, so the later Rev E DDR configuration changes are not part of this camera-only setup.

## 2. Source design and attribution

The camera implementation was adapted from the OV7670 ZedBoard example published at:

**Alex Zhang, OV7670 Guide**  
https://github.com/Alexzhang1988/OV7670_guide

The source project is based primarily on earlier OV7670 FPGA work by **Mike Field**. Original attribution comments are preserved in the VHDL source files.

The camera register file also notes that some settings were derived from the Linux kernel `OV7670.c` driver.

The **OmniVision OV7670/OV7171 Implementation Guide** was used as supporting documentation for camera timing, control signals, register behavior, and SCCB configuration.

## 3. Camera-only project files

The camera-only Vivado project uses the following VHDL files:

| File | Purpose |
|---|---|
| `ov7670_top.vhd` | Top-level camera/VGA design |
| `ov7670_controller.vhd` | Camera initialization controller |
| `ov7670_registers.vhd` | OV7670 register configuration sequence |
| `ov7670_capture.vhd` | Captures camera pixels and creates frame-buffer write data |
| `i2c_sender.vhd` | Sends camera configuration commands over the I2C-like SCCB interface |
| `vga.vhd` | Generates 640×480 VGA timing and frame-buffer read addresses |
| `debounce.vhd` | Debounces the camera reinitialization push button |
| `clocking.vhd` | Generates the 50 MHz and 25 MHz clocks from the 100 MHz board clock |

The project also uses:

| File | Purpose |
|---|---|
| `ZedBoard.xdc` | Physical pin assignments and I/O standards |
| `blk_mem_gen_0.xci` | Exact Vivado Block Memory Generator configuration used as the frame buffer |

The top-level design contains no neural-network, test-ROM, processed-image, or dataset-specific components.

## 4. Baseline camera architecture

The camera-only project implements this data path:

```text
                        100 MHz ZedBoard Clock
                                 |
                                 v
                              clocking
                           /             \
                       50 MHz           25 MHz
                         |                 |
                         v                 v
                 OV7670 Controller       VGA
                         |                 ^
                         v                 |
                    OV7670 Camera         |
                         |                 |
             PCLK / VSYNC / HREF / D[7:0]
                         |
                         v
                  OV7670 Capture
                         |
             19-bit address / 12-bit pixel
                         |
                         v
               Dual-Port Block RAM
                  ^               |
                  |               |
             OV7670 PCLK       25 MHz VGA
               write              read
```

The Block RAM provides the clock-domain boundary between camera acquisition using `OV7670_PCLK` and VGA display using the generated 25 MHz clock.

## 5. Clock generation

The `clocking.vhd` source generates two clocks from the ZedBoard 100 MHz input clock:

- **50 MHz** for camera configuration/control; and
- **25 MHz** for VGA timing.

The top-level design supplies the 50 MHz clock to `ov7670_controller` and the 25 MHz clock to the VGA module.

The OV7670 supplies its own `PCLK` for incoming pixel capture. This means the camera write side of the frame buffer and the VGA read side operate in separate clock domains.

## 6. OV7670 interface signals

The camera connection uses three main signal groups.

### Configuration and control

- `OV7670_SIOC`
- `OV7670_SIOD`
- `OV7670_RESET`
- `OV7670_PWDN`
- `OV7670_XCLK`

### Camera timing

- `OV7670_PCLK`
- `OV7670_VSYNC`
- `OV7670_HREF`

### Pixel data

- `OV7670_D[7:0]`

The OmniVision documentation identifies `VSYNC` as the vertical synchronization signal, `HREF` as the horizontal data-valid reference, and `PCLK` as the pixel clock used to capture parallel camera data.

## 7. Physical pin assignments

The camera-only project constraints file, `ZedBoard.xdc`, is the authoritative source for FPGA package-pin assignments.

The camera signals are assigned as follows:

| Camera signal | Zynq package pin |
|---|---|
| `OV7670_PWDN` | Y11 |
| `OV7670_RESET` | AB11 |
| `OV7670_D[0]` | AA11 |
| `OV7670_D[1]` | AB10 |
| `OV7670_D[2]` | Y10 |
| `OV7670_D[3]` | AB9 |
| `OV7670_D[4]` | AA9 |
| `OV7670_D[5]` | AA8 |
| `OV7670_D[6]` | W12 |
| `OV7670_D[7]` | V12 |
| `OV7670_XCLK` | W11 |
| `OV7670_PCLK` | W10 |
| `OV7670_HREF` | V10 |
| `OV7670_VSYNC` | V9 |
| `OV7670_SIOD` | W8 |
| `OV7670_SIOC` | V8 |

The constraints file also defines the ZedBoard VGA pins, LEDs, 100 MHz input clock, push button, and I/O standards used by the design.

## 8. Physical wiring documentation

The following photographs document the camera-to-ZedBoard wiring used during development.

### ZedBoard header wiring

![Close-up of ZedBoard camera header wiring](images/camera_zedboard_header_wiring.jpg)

### ZedBoard wiring overview

![ZedBoard with camera jumper wiring](images/camera_zedboard_overview.jpg)

### OV7670 connection overview

![OV7670 camera connected to the ZedBoard](images/ov7670_wiring_overview.jpg)

### OV7670 pin labels

![OV7670 module showing labeled pins](images/ov7670_pin_labels.jpg)

The handwritten notes below record the wiring work performed during bring-up.

![Handwritten camera wiring notes](images/camera_wiring_notes.jpg)

The final physical jumper placement is shown here:

![Final OV7670-to-ZedBoard wiring placement](images/final_camera_wiring_placement.jpeg)

For reproduction, use `ZedBoard.xdc` for the electrical signal-to-package-pin mapping and the photographs as physical wiring references.

## 9. Camera capture and frame-buffer format

`ov7670_capture.vhd` receives:

- the OV7670 pixel clock;
- `VSYNC`;
- `HREF`;
- and the 8-bit camera data bus.

The capture logic combines the incoming camera bytes and writes a **12-bit RGB pixel value** to the frame buffer.

The camera-only implementation uses a **19-bit frame address** on both the capture and VGA sides. This is the baseline frame-buffer implementation and should not be confused with later reduced-resolution or dataset-specific addressing.

## 10. Dual-port Block RAM

The top-level design instantiates `blk_mem_gen_0` as a dual-port frame buffer.

### Write port

The camera capture path writes using:

```text
clock   = OV7670_PCLK
address = capture_addr[18:0]
data    = capture_data[11:0]
write   = capture_we
```

### Read port

The VGA path reads using:

```text
clock   = 25 MHz
address = frame_addr[18:0]
data    = frame_pixel[11:0]
```

The exact frame-buffer IP configuration is stored in `blk_mem_gen_0.xci` and should be preserved with the camera-only source code.

## 11. VGA display path

`vga.vhd` generates standard **640×480** VGA timing using the 25 MHz clock.

The 12-bit frame-buffer pixel is divided into:

```text
frame_pixel[11:8] -> VGA red
frame_pixel[7:4]  -> VGA green
frame_pixel[3:0]  -> VGA blue
```

The VGA module generates the frame-buffer read addresses and outputs black pixels during the blanking intervals.

## 12. Bring-up diagnostics

The camera-only top level uses ZedBoard LEDs as basic diagnostic indicators:

- `LED[0]` indicates that camera configuration completed;
- `LED[1]` is driven from a counter clocked by `OV7670_PCLK`, providing visible evidence of camera pixel-clock activity;
- `LED[3]` is driven from a counter on the generated system clock as a general activity indicator.

The push button is passed through `debounce.vhd` and can restart the camera register-configuration sequence.

These signals were useful for distinguishing a configuration or clock problem from a VGA-display problem during camera bring-up.

## 13. Functional validation and troubleshooting

The camera was validated incrementally.

### Initial corrupted VGA output

![Initial striped/corrupted VGA camera output](images/vga_initial_corrupted_output.jpg)

The structured but corrupted display showed that data were reaching the VGA path, while the complete camera/frame-buffer/display sequence was not yet operating correctly.

### Partial valid image

![Small valid camera image during intermediate testing](images/vga_partial_camera_output.jpg)

A recognizable image region demonstrated progress toward a working capture and display path.

### Working live camera feed

![Working OV7670 camera feed displayed through VGA](images/vga_working_camera_output.jpg)

The baseline was considered functional when the monitor displayed a recognizable live image from the OV7670. The printed digit shown in the final photograph was simply an easy visual target; no MNIST-specific preprocessing or classification was being performed at this stage.

## 14. Baseline completion criteria

The generic camera setup was complete once:

- the OV7670 register sequence completed;
- the camera pixel clock was active;
- `VSYNC`, `HREF`, and parallel pixel data were accepted by the capture logic;
- captured pixels were written to the dual-port frame buffer;
- the VGA path could read that frame buffer independently; and
- a recognizable live camera image appeared on the VGA monitor.

This baseline provided a known-working camera interface before dataset-specific image processing or NN hardware was introduced.

## 15. Reproduction checklist

To recreate the camera-only project:

1. Create a Vivado 2018.1 project for `XC7Z020CLG484-1`.
2. Add the eight camera-only VHDL source files.
3. Set `ov7670_top` as the top-level entity.
4. Add the camera-only `ZedBoard.xdc`.
5. Add `blk_mem_gen_0.xci` from the camera-only Vivado project.
6. Verify that the 100 MHz clock source and generated 50/25 MHz clock logic are present.
7. Connect the OV7670 according to the constraints and camera module labels.
8. Connect the ZedBoard VGA output to a display.
9. Synthesize, implement, and generate the bitstream.
10. Program the ZedBoard.
11. Confirm configuration completion and camera-clock activity using the LEDs.
12. Confirm that a recognizable camera image is displayed through VGA.

No Zynq processing-system DDR configuration is required for this baseline because the camera frame is stored in programmable-logic Block RAM.

## 16. Relationship to later FPGA work

This notebook ends with the working generic camera/VGA pipeline:

```text
OV7670 -> Capture -> Frame BRAM -> VGA
```

Dataset-specific branches extend this baseline with their own preprocessing and NN input paths. Keeping those later changes separate makes the camera implementation reusable across the FPGA experiments.

## Repository image files

This notebook expects the following documentation images in an `images/` directory:

```text
camera_zedboard_header_wiring.jpg
camera_zedboard_overview.jpg
ov7670_wiring_overview.jpg
ov7670_pin_labels.jpg
camera_wiring_notes.jpg
final_camera_wiring_placement.jpeg
vga_initial_corrupted_output.jpg
vga_partial_camera_output.jpg
vga_working_camera_output.jpg
```